# 01. 데이터 로드 및 전처리

> 이 노트북은 원본 통합 노트북 `소스_코드_8팀_생체_신호_기반_흡연_여부_비교_시각화_프로젝트__-_종합.ipynb` 의 관련 셀들을 주제별로 재구성한 것입니다. 코드는 원본 그대로 옮겨왔으며(실행 결과만 초기화), 새로 작성하거나 수정한 로직은 없습니다.
>
> 실행하려면 `train_dataset.csv`(Kaggle: Smoker Status Prediction Dataset)를 동일 폴더에 두어야 합니다. 데이터 파일은 저장소에 포함되어 있지 않습니다.
>
> 원본에는 서로 다른 시점에 작성된 3개의 전처리 버전이 섞여 있습니다 (자세한 내용은 `docs/03_issues_and_troubleshooting.md` 참고). 이 재구성본은 그 버전들을 **삭제하거나 하나로 합치지 않고**, 각 노트북 안에서 '버전 A/B/C'로 구분해 모두 보존했습니다.

이 노트북은 원본에 섞여 있던 3개의 전처리 버전을 모두 담고 있습니다.

## 버전 A — 초기 탐색 (이상치 기준: 시력 ≤ 3.0, BMI 5구간)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from math import pi

In [ ]:
!pip install koreanize-matplotlib
import koreanize_matplotlib

In [ ]:
C_SMOKER = '#A63A50'
C_HIGH_BMI = '#C65D6E'
C_NONSMOKER = '#6B7075'
C_LOW_BMI = '#2F3437'
C_TEXT = '#1E1E1E'
C_GRID = '#D9D9D9'

sns.set_palette([C_NONSMOKER, C_SMOKER])
plt.rcParams.update({
    "text.color": C_TEXT,
    "axes.labelcolor": C_TEXT,
    "xtick.color": C_TEXT,
    "ytick.color": C_TEXT
})

In [ ]:
df = pd.read_csv('/content/train_dataset.csv')

In [ ]:
df.head()

In [ ]:
df.info() # 결측치 없음

In [ ]:
df.duplicated().sum()
train = df.drop_duplicates()
train.duplicated().sum()

In [ ]:
train.describe() #시력 9.9, ALT 2914등 이상치 값이 많다는 사실을 알 수 있다.

In [ ]:
train = df[
    (df['Gtp'] < 500) &
    (df['ALT'] < 500) &
    (df['AST'] < 500) &
    (df['LDL'] < 400) &
    (df['serum creatinine'] < 3.0) &
    (df['eyesight(left)'] <= 3.0) &
    (df['eyesight(right)'] <= 3.0)
].copy()
train.describe()
# 이상치 제거 완료

In [ ]:
train['BMI'] = train['weight(kg)']/(train['height(cm)']/100)**2
train['BMI_group'] =pd.cut(train['BMI'],
                                 bins=[0, 18.5, 23, 25, 30, 100],
                                 labels=['저체중', '정상', '과체중', '비만2단계', '고도비만'])
train['BMI_group']
#BMI 분류하기

## 버전 B — 나눔 폰트 설치 및 회귀분석용 정리본 (high_BMI 이진변수, 연령대 6구간)

In [ ]:
!apt-get update -qq
!apt-get install -y fonts-nanum -qq

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

nanum_fonts = sorted(set(
    f.name for f in fm.fontManager.ttflist
    if 'Nanum' in f.name
))
print(nanum_fonts)

In [ ]:
!ls /usr/share/fonts/truetype/nanum

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fm.fontManager.addfont(font_path)
font_name = fm.FontProperties(fname=font_path).get_name()

plt.rcParams['font.family'] = font_name
plt.rcParams['axes.unicode_minus'] = False

print("사용 폰트:", font_name)

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import mannwhitneyu
import statsmodels.formula.api as smf

COLOR_SMOKER = '#A63A50'
COLOR_HIGH_BMI = '#C65D6E'
COLOR_NONSMOKER = '#6B7075'
COLOR_LOW_BMI = '#2F3437'
COLOR_TEXT = '#1E1E1E'
COLOR_GRID = '#D9D9D9'

plt.rcParams['axes.unicode_minus'] = False


train = pd.read_csv('/content/train_dataset.csv')


train_clean = train[
    (train['Gtp'] < 500) &
    (train['ALT'] < 500) &
    (train['AST'] < 500) &
    (train['LDL'] < 400) &
    (train['serum creatinine'] < 3.0) &
    (train['eyesight(left)'] <= 3.0) &
    (train['eyesight(right)'] <= 3.0)
].copy()

# BMI 생성
train_clean['BMI'] = train_clean['weight(kg)'] / ((train_clean['height(cm)'] / 100) ** 2)

# 고BMI 변수 생성
train_clean['high_BMI'] = (train_clean['BMI'] >= 25).astype(int)

# 연령대 변수 생성
train_clean['age_group'] = pd.cut(
    train_clean['age'],
    bins=[15, 29, 39, 49, 59, 69, 120],
    labels=['20대', '30대', '40대', '50대', '60대', '70대+']
)

print("원본 데이터 크기:", train.shape)
print("전처리 후 데이터 크기:", train_clean.shape)
display(train_clean.head())
display(train_clean.describe())

## 버전 C — 발표 반영본 (이상치 기준: 시력 ≤ 2.0, BMI 6구간, WHtR 추가)

In [ ]:
!sudo apt-get install -y fonts-nanum                  # 나눔 폰트 설치
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math as ma

In [ ]:
import matplotlib.pyplot as plt

plt.rc('font', family='NanumBarunGothic')           # 한글 폰트 설정 테스트
plt.rcParams['axes.unicode_minus'] = False

plt.figure(figsize=(5, 3))                          # 설치 확인
plt.text(0.5, 0.5, '한글 폰트 설치 성공!', size=20, ha='center')
plt.title('폰트 테스트')
plt.show()

In [ ]:
# 1) CSV 파일을 읽어 DataFrame 생성
manu_path = '/content/train_dataset.csv'        # train_dataset.csv 파일 경로 입력

df = pd.read_csv(manu_path)


df.head(5)                                      # df의x 상위 5개 행을 확인하고 출력

In [ ]:
df['BMI'] = df['weight(kg)'] / ((df['height(cm)'] / 100) ** 2)        # BMI 추가
df['WHtR'] = df['waist(cm)'] / df['height(cm)']                       # WHtR 추가

display(df[['height(cm)', 'weight(kg)', 'waist(cm)', 'BMI', 'WHtR']].head())

In [ ]:
# 필터적용
train_clean = df[
    (df['Gtp'] < 500) &
    (df['ALT'] < 500) &
    (df['AST'] < 500) &
    (df['LDL'] < 400) &
    (df['serum creatinine'] < 3.0) &
    (df['eyesight(left)'] <= 2.0) &
    (df['eyesight(right)'] <= 2.0)
].copy()
train_clean.describe()

In [ ]:
# 나이를 20-35를 '청년', 36-50을 '중년', 51-64 를 '장년', 65 이상 '노년'으로 해서 나이를 4 그룹으로 분류한 값 추가
bins = [0, 35, 50, 64, 150]
labels = ['청년', '중년', '장년', '노년']

train_clean['age_group'] = pd.cut(train_clean['age'], bins=bins, labels=labels, right=True)

display(train_clean[['age', 'age_group']].head())

In [ ]:
# BMI 분류 및 항목 추가
bmi_bins = [0, 18.5, 23, 25, 30, 35, 100]
bmi_labels = ['저체중', '정상', '과체중', '1단계 비만', '2단계 비만', '3단계 비만']

train_clean['bmi_group'] = pd.cut(train_clean['BMI'], bins=bmi_bins, labels=bmi_labels, right=False)

display(train_clean[['BMI', 'bmi_group']].head())